# 12. Logical meshes

The model of global semantics on top of the L1 core of notebooks 1-11: a **logical mesh** is the whole mesh, with global indices. It holds what a mesh *means*: its topology (the tables), its semantic regions, its geometry, and an identity. Its decomposition onto ranks (who owns which points) is separate configuration, used when a global program is lowered to local programs (see "13. Global programs"). Fields know their logical mesh: they read through its tables without a `with` block, and are combined only with fields of the same mesh.

The example: a plate of 4 × 2 cells, cell `c = 4 j + i` at column `i` and row `j`, with two regions, `West` (the cells 0, 4) and `East` (the cells 3, 7), and a vertical axis `K` of 2 levels.

```
  +---+---+---+---+
  | 4 | 5 | 6 | 7 |      West: 0, 4
  +---+---+---+---+      East: 3, 7
  | 0 | 1 | 2 | 3 |
  +---+---+---+---+
```

Contents: schemas and locations · the logical mesh · fields on the mesh · roles and the compile key.

## Schemas and locations

| Concept | Kind | Meaning |
|---|---|---|
| `Schema` | class | A brand for the locations of its meshes, declared by subclassing: `class Plate(Schema): ...`. A type, never instantiated |
| `Face[M]`, `Edge[M]`, `Node[M]` | dimension | The families of a schema, e.g. `Face[Plate]`: UGRID's 2-, 1- and 0-dimensional locations, e.g. ICON's cells, edges and vertices. Each is a `Location`, one class per family and schema, created on first use |
| `SchemaLocation` | class | The base of the families: a `Location` with a `schema` |
| `location_of` | function | `location_of(Face, Plate)` is `Face[Plate]`, for a family and a schema only known at runtime |

A family is an ordinary L1 `Location`: relations, regions and fields are declared on it as in the other notebooks. An alias names it: `Cell = Face[Plate]`, in annotations and at runtime.

**Roles** are subclasses of a schema, e.g. `class Coarse(Plate): ...` and `class Fine(Plate): ...` for two resolutions of one grid. `Face[Coarse]` and `Face[Fine]` are distinct locations, so each role declares its own relations. At runtime `Face[Coarse]` is not a subclass of `Face[Plate]`; type checkers, however, accept a `Face[Coarse]` field where a `Face[Plate]` one is expected.

In [1]:
import contextlib
import io

import numpy as np
from concepts import Axis, Edge, Face, Location, Node, Schema, SchemaLocation, location_of


class Plate(Schema): ...


Cell = Face[Plate]  # the cells of the plate


class K(Axis): ...  # vertical levels


print(
    "Cell:",
    Cell.__name__,
    "| schema:",
    Cell.schema.__name__,
    "| Face[Plate] is Cell:",
    Face[Plate] is Cell,
)
print(
    "a Location:",
    issubclass(Cell, Location),
    "| a SchemaLocation:",
    issubclass(Cell, SchemaLocation),
    "| location_of(Face, Plate) is Cell:",
    location_of(Face, Plate) is Cell,
)
print(
    "other families:",
    Edge[Plate].__name__,
    Node[Plate].__name__,
    "| Edge[Plate] is Cell:",
    Edge[Plate] is Cell,
)


class Coarse(Plate): ...  # two roles of the schema Plate


class Fine(Plate): ...


print(
    "roles:",
    Face[Coarse].__name__,
    Face[Coarse].schema.__name__,
    "| Face[Coarse] is Face[Fine]:",
    Face[Coarse] is Face[Fine],
    "| a subclass of Face[Plate]:",
    issubclass(Face[Coarse], Face[Plate]),
)


class Cell0(Location): ...  # a plain L1 location


try:
    Face[Cell0]
except TypeError as error:
    print("TypeError:", error)
try:
    Plate()
except TypeError as error:
    print("TypeError:", error)

Cell: Face[Plate] | schema: Plate | Face[Plate] is Cell: True
a Location: True | a SchemaLocation: True | location_of(Face, Plate) is Cell: True
other families: Edge[Plate] Node[Plate] | Edge[Plate] is Cell: False
roles: Face[Coarse] Coarse | Face[Coarse] is Face[Fine]: False | a subclass of Face[Plate]: False
TypeError: 'Cell0' is not a schema: subclass `Schema`, e.g. `class Icon(Schema): ...`.
TypeError: A schema is a type: it is never instantiated.


## The logical mesh

| Concept | Kind | Meaning |
|---|---|---|
| `LogicalMesh` | class (a `Mesh`) | `LogicalMesh(schema, tables, extents=..., regions=..., geometry=..., mesh_id=...)`: the whole mesh, with global indices: its `tables`, the `extents` of its locations and axes, its `regions` (global index sets of `Region[D]` classes), its `geometry` (named fields, e.g. the cell areas) and its identity, the `mesh_id` (by default `"<schema>-<n>"`, counting the logical meshes). Immutable |
| `domain` | method | `grid.domain(Cell, K)`: the whole domain of the given dimensions |
| `region` | method | `grid.region(East)`: the global domain of a region |
| `signature` | property | The compile key: the schema, the relations and the regions (see the last section) |

The checks, when the mesh is built:
- the locations of the tables, regions and extents are families of the schema, and each has an extent (a `ValueError`);
- the tables are checked as for any `Mesh`, against the extents;
- the points of each region are sorted and within the extent (a `ValueError`);
- each geometry field is on the whole of its dimensions (a `ValueError`).

A logical mesh is still a `Mesh`: `grid[C2C]` is a table, and `with grid:` provides the tables to code that reads plain fields.

In [2]:
from concepts import MISSING_VALUE, LogicalMesh, Mesh, Region, Relation
from implementation import NumPyArrayField, format_domain, neighbor_table


class C2C(Relation[Cell, Cell], max_neighbors=4, partial=True): ...  # [west, east, south, north]


class West(Region[Cell]): ...


class East(Region[Cell]): ...


def at(i, j):  # the cell at column i and row j, or missing
    return 4 * j + i if 0 <= i < 4 and 0 <= j < 2 else MISSING_VALUE


rows = [
    [at(i - 1, j), at(i + 1, j), at(i, j - 1), at(i, j + 1)] for j in range(2) for i in range(4)
]
plate = {C2C: neighbor_table(C2C, rows)}
area = NumPyArrayField({Cell: range(8)}, [1.0, 1.0, 1.0, 1.0, 2.0, 2.0, 2.0, 2.0])  # row 1: wider
grid = LogicalMesh(
    Plate,
    plate,
    extents={Cell: 8, K: 2},
    regions={West: (0, 4), East: (3, 7)},
    geometry={"area": area},
)
print(grid, "| mesh_id:", grid.mesh_id)  # Plate-0: the first logical mesh of this kernel
print("domain:", format_domain(grid.domain(Cell, K)), "| East:", format_domain(grid.region(East)))
# cell 0: no west, east 1, no south, north 4; cell 5: west 4, east 6, south 1, no north
print("C2C rows 0, 5:", grid[C2C].data[[0, 5]].tolist())
print("a Mesh:", isinstance(grid, Mesh), "| relations:", [r.__name__ for r in grid])


class Plain(Relation[Cell0, Cell0], max_neighbors=1): ...  # on a plain L1 location


for build in (
    lambda: LogicalMesh(Plate, {Plain: neighbor_table(Plain, [[0]])}, extents={Cell0: 1}),
    lambda: LogicalMesh(Plate, plate, extents={Cell: 8}, regions={East: (3, 8)}),  # no cell 8
):
    try:
        build()
    except ValueError as error:
        print("ValueError:", error)
try:
    grid.mesh_id = "Plate-7"
except AttributeError as error:
    print("AttributeError:", error)

LogicalMesh('Plate-0') | mesh_id: Plate-0
domain: Face[Plate][0:8], K[0:2] | East: Face[Plate]{3, 7}
C2C rows 0, 5: [[-2, 1, -2, 4], [4, 6, 1, -2]]
a Mesh: True | relations: ['C2C']
ValueError: 'Cell0' is not a location of the schema 'Plate': use `Face[Plate]`, `Edge[Plate]` or `Node[Plate]`.
ValueError: The points of the region 'East' must be within the extent of 'Face[Plate]', 8.
AttributeError: A `LogicalMesh` is immutable.


## Fields on the mesh

| Concept | Kind | Meaning |
|---|---|---|
| `MeshField` | protocol | An `NDArrayField` on a logical mesh, the pair (`mesh`, values), with global indices |
| `zeros`, `full`, `field` | methods | `grid.zeros(Cell, K)`, `grid.full(Cell, fill_value=1.0)`, `grid.field(Cell, data=[...])`: a `MeshField` on the whole of the given dimensions |
| `geometry` | attribute | The geometry fields of the mesh, by name, as `MeshField`s: `grid.geometry["area"]` |

`NumPyMeshField` implements it, and `mesh_field(grid, domain, data)` builds one on any domain of the mesh.

- **Its own tables.** Premaps of a mesh field, e.g. `t(C2C)`, its neighbor reductions (`sum_over`, `max_over`, `min_over`), `flatten` and `unflatten` use the tables of its mesh: no `with` block, and another mesh's `with` block does not change them. A DSL operator called with mesh fields runs in their mesh.
- **The brand.** Arithmetic and writes combine a mesh field only with fields of the same mesh, the same `mesh_id` (else a `ValueError`), or with plain fields; DSL operators check their field arguments the same way, before computing. Results keep the brand: arithmetic, subscripts, premaps, reductions and the results of DSL operators, tuples of them too. The lazy backends return `DeferredField`s, which are not branded, so the brand is not checked on them.

In [3]:
from concepts import Dims, Field, MeshField, Neighbor
from implementation import eager, field_operator, max_over, sum_over

t = grid.field(Cell, data=np.arange(8.0))  # e.g. a temperature, 0 to 7
print("t =", t, "| a MeshField:", isinstance(t, MeshField), "| t.mesh is grid:", t.mesh is grid)
near = t(C2C.boundary(0.0))  # no `with` block: the tables of t's mesh
print("near:", format_domain(near.domain), "| a MeshField:", isinstance(near, MeshField))
# cell 0: east 1 + north 4 = 5; cell 5: west 4 + east 6 + south 1 = 11; cell 6: 5 + 7 + 2 = 14
print("sum_over =", sum_over(near, axis=Neighbor[C2C]))
# cell 0: max(0, 1, 0, 4) = 4 (0 for a missing neighbor); cell 3: max(2, 0, 0, 7) = 7
print("max_over =", max_over(near, axis=Neighbor[C2C]))


@field_operator()
def around(t: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    return sum_over(t(C2C.boundary(0.0)), axis=Neighbor[C2C])


with contextlib.redirect_stdout(io.StringIO()):  # silence the backend
    result = eager(around)(t)  # no `with` block either
print("around(t) =", result, "| a MeshField:", isinstance(result, MeshField))
zeros = grid.zeros(Cell, K)
print(
    "zeros:",
    format_domain(zeros.domain),
    "| all 0.0:",
    (zeros.data == 0.0).all(),
    "| a MeshField:",
    isinstance(zeros, MeshField),
)
print("full in West =", grid.full(Cell, fill_value=2.5)[grid.region(West)])  # cells 0, 4
# the area times t: row 0, 1 x (0, 1, 2, 3); row 1, 2 x (4, 5, 6, 7) = 8, 10, 12, 14
print("area * t =", grid.geometry["area"] * t)
print("t in East =", t[grid.region(East)])  # cells 3 and 7

t = Field(Face[Plate][0:8]: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]) | a MeshField: True | t.mesh is grid: True
near: Face[Plate][0:8], Neighbor[C2C][0:4] | a MeshField: True
sum_over = Field(Face[Plate][0:8]: [5.0, 7.0, 10.0, 9.0, 5.0, 11.0, 14.0, 9.0])
max_over = Field(Face[Plate][0:8]: [4.0, 5.0, 6.0, 7.0, 5.0, 6.0, 7.0, 6.0])
around(t) = Field(Face[Plate][0:8]: [5.0, 7.0, 10.0, 9.0, 5.0, 11.0, 14.0, 9.0]) | a MeshField: True
zeros: Face[Plate][0:8], K[0:2] | all 0.0: True | a MeshField: True
full in West = Field(Face[Plate]{0, 4}: [2.5, 2.5])
area * t = Field(Face[Plate][0:8]: [0.0, 1.0, 2.0, 3.0, 8.0, 10.0, 12.0, 14.0])
t in East = Field(Face[Plate]{3, 7}: [3.0, 7.0])


In [4]:
other = LogicalMesh(Plate, plate, extents={Cell: 8})  # the same tables: another mesh, Plate-1
plain = NumPyArrayField({Cell: range(8)}, np.ones(8))  # a plain field, without a mesh
print("other:", other.mesh_id, "| t + plain:", t + plain, "| branded:", (plain + t).mesh is grid)


@field_operator()
def difference(
    a: Field[Dims[Cell], float], b: Field[Dims[Cell], float]
) -> Field[Dims[Cell], float]:
    return a - b


for combine in (
    lambda: t + other.field(Cell, data=np.ones(8)),
    lambda: difference(t, other.zeros(Cell)),  # checked before computing
):
    try:
        combine()
    except ValueError as error:
        print("ValueError:", error)
try:
    t[...] = other.zeros(Cell)
except ValueError as error:
    print("ValueError:", error, "| t unchanged:", t.data.tolist() == list(range(8)))

other: Plate-1 | t + plain: Field(Face[Plate][0:8]: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]) | branded: True
ValueError: The fields are on different logical meshes, 'Plate-0' and 'Plate-1'.
ValueError: The fields are on different logical meshes, 'Plate-0' and 'Plate-1'.
ValueError: The fields are on different logical meshes, 'Plate-0' and 'Plate-1'. | t unchanged: True


## Roles and the compile key

Several logical meshes can share a schema: e.g. the same grid at two resolutions, or two experiments on one grid. A generic program only depends on their **signature**: the schema, the relations of the tables and the regions, as classes, sorted by name. Not the instance, its `mesh_id`, the extents or the values: meshes with one signature share their compiled operators (see "13. Global programs").

A role has its own locations, so its meshes have other signatures, and it declares its own relations: a relation on `Face[Plate]` cannot be in a mesh of the role `Coarse`.

In [5]:
def plate_rows(ni, nj):  # the C2C rows of a plate of ni x nj cells, cell c = ni j + i
    def at(i, j):
        return ni * j + i if 0 <= i < ni and 0 <= j < nj else MISSING_VALUE

    return [
        [at(i - 1, j), at(i + 1, j), at(i, j - 1), at(i, j + 1)]
        for j in range(nj)
        for i in range(ni)
    ]


def names(signature):
    schema, relations, regions = signature
    return schema.__name__, [r.__name__ for r in relations], [r.__name__ for r in regions]


print("grid:", names(grid.signature))
# a plate of 4 x 3 cells: other tables, extents and region points; the same relations and regions
grid2 = LogicalMesh(
    Plate,
    {C2C: neighbor_table(C2C, plate_rows(4, 3))},
    extents={Cell: 12},
    regions={East: (3, 7, 11), West: (0, 4, 8)},  # given in another order: sorted by name
)
print("grid2:", grid2.mesh_id, "| same signature:", grid.signature == grid2.signature)
grid3 = LogicalMesh(Plate, plate, extents={Cell: 8})  # no regions
print("grid3:", names(grid3.signature), "| same signature:", grid.signature == grid3.signature)

CoarseCell = Face[Coarse]  # the role Coarse declares its own relation


class CoarseC2C(Relation[CoarseCell, CoarseCell], max_neighbors=4, partial=True): ...


coarse = LogicalMesh(
    Coarse, {CoarseC2C: neighbor_table(CoarseC2C, plate_rows(2, 1))}, extents={CoarseCell: 2}
)
print("coarse:", names(coarse.signature), "| same signature:", grid.signature == coarse.signature)
try:
    LogicalMesh(Coarse, plate, extents={Cell: 8})  # C2C is a relation of Face[Plate]
except ValueError as error:
    print("ValueError:", error)

grid: ('Plate', ['C2C'], ['East', 'West'])
grid2: Plate-2 | same signature: True
grid3: ('Plate', ['C2C'], []) | same signature: False
coarse: ('Coarse', ['CoarseC2C'], []) | same signature: False
ValueError: 'Face[Plate]' is not a location of the schema 'Coarse': use `Face[Coarse]`, `Edge[Coarse]` or `Node[Coarse]`.


## Summary

| L1 (notebooks 1-11) | L2, the logical mesh |
|---|---|
| `class Cell(Location): ...` | `Cell = Face[Plate]`: a family of a schema; roles, `Face[Coarse]`, are other locations |
| `Mesh(tables, extents=...)`, provided by a `with` block | `LogicalMesh(Plate, tables, extents=..., regions=..., geometry=...)`: immutable, with an identity, the `mesh_id` |
| `NumPyArrayField(domain, data)`, reading the current mesh | `grid.zeros`, `grid.full`, `grid.field`: `MeshField`s, reading their own mesh |
| a `LocalMesh` per rank, with its regions on local points | `grid.region(East)`: global regions; the decomposition is configuration (see "13. Global programs") |
| any fields combine | fields of different meshes do not (a `ValueError`) |
| | `signature`: the compile key, shared by meshes of one schema, relations and regions |